# Human-in-the-Loop: Applied Patterns

This notebook builds on the interrupt/resume mechanics from **`01_HITL_Mechanics.ipynb`** and applies them to real, LLM-backed agent workflows:

1. **Tool-calling agent with `interrupt_before`** — pause before the assistant node, inspect/edit state, resume.
2. **The Approve/Reject pattern** — `interrupt()` + `Command(goto=...)` to branch on a human decision.
3. **The Edit/Review pattern** — let a human edit LLM output before it's used downstream.
4. **Reviewing tool calls before execution** — `interrupt()` inside a tool, plus a generic wrapper that adds review to any tool.

All sections share one LLM instance, initialized once below.

## Shared Setup

In [ ]:
# ============================================================================
# LLM SETUP: shared across all HITL patterns in this notebook
# ============================================================================
from dotenv import load_dotenv
load_dotenv()

import os
import sys
sys.path.append(os.path.abspath("../.."))
from helpers import get_llm

llm = get_llm()
print(f"LLM initialized: {llm}")

## 1. Tool-Calling Agent with Human-in-the-Loop (`interrupt_before`)

Goals of human-in-the-loop for an agent:

1. **Approval** — interrupt the agent, surface state to a user, and allow them to accept an action.
2. **Debugging** — rewind the graph to reproduce or avoid issues.
3. **Editing** — modify the state before the agent continues.

LangGraph offers several ways to get or update agent state to support these workflows.

In [ ]:
# Custom tools
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a by b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]
tools

In [ ]:
# Integrate tools with the LLM
llm_with_tools = llm.bind_tools(tools)
llm_with_tools

### Build the Graph, Interrupting Before the Assistant Node

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Assistant node
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# Graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message from assistant is a tool call -> tools_condition routes to tools
    # If it's not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()

# Human-in-the-loop: pause before every assistant turn
graph = builder.compile(interrupt_before=["assistant"], checkpointer=memory)

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
thread = {"configurable": {"thread_id": "123"}}
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}

for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

In [ ]:
state = graph.get_state(thread)
state.next  # ('assistant',) -- the graph is paused right before the assistant runs

### Resume Execution

Passing `None` as input resumes the graph from its last checkpoint.

In [ ]:
# Continue the execution to Assistant -> Tools
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

In [ ]:
state = graph.get_state(thread)
state.next  # paused again before the next assistant turn

In [ ]:
# Continue the execution of Assistant and then end
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

### Variation: Editing Human Feedback Mid-Run

Same graph, a fresh thread — while the graph is paused before `assistant`, we call `graph.update_state(...)` to replace the human's message before letting the agent continue.

In [ ]:
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}
thread = {"configurable": {"thread_id": "1"}}

for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

In [ ]:
state = graph.get_state(thread)
state.next

In [ ]:
# Edit the state before the assistant sees it
graph.update_state(thread, {"messages": [HumanMessage(content="No, please multiply 15 and 6")]})

In [ ]:
new_state = graph.get_state(thread).values
for m in new_state['messages']:
    m.pretty_print()

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

### Variant: A Graph That Waits for User Input via a Dedicated `human_feedback` Node

Instead of interrupting before `assistant`, this graph inserts an explicit no-op `human_feedback` node and interrupts before *that* — a slightly different shape that makes the pause point self-documenting in the graph diagram, and lets you resume with `as_node="human_feedback"` to inject the human's message at exactly that point.

In [ ]:
def human_feedback(state: MessagesState):
    pass

# Graph with a dedicated human_feedback node
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_node("human_feedback", human_feedback)

builder.add_edge(START, "human_feedback")
builder.add_edge("human_feedback", "assistant")
builder.add_conditional_edges(
    "assistant",
    tools_condition,
)
builder.add_edge("tools", "human_feedback")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["human_feedback"], checkpointer=memory)

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_input = {"messages": "Multiply 2 and 3"}
thread = {"configurable": {"thread_id": "5"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

# Get user input and inject it at the human_feedback node
user_input = input("Tell me how you want to update the state:")
graph.update_state(thread, {"messages": user_input}, as_node="human_feedback")

In [ ]:
# Continue the graph execution
for event in graph.stream(None, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

## 2. The Approve/Reject Pattern

**How `interrupt()` works here:**
1. **Pauses execution** — the graph stops and returns control to the caller.
2. **Returns context** — a dict with the question and relevant data for human review.
3. **Awaits input** — the graph waits for human input to be supplied on resume.

**The `decision` variable** receives whatever value was passed in `Command(resume="...")`. Calling `graph.invoke(Command(resume="approve"), config=config)` makes `decision == "approve"` inside the node.

**`Command`** is a control-flow object with two jobs:
- `goto`: jump to a specific named node
- `update`: update the shared state

The type annotation `Command[Literal["approved_path", "rejected_path"]]` documents exactly which nodes this command can navigate to.

In [ ]:
from typing import Literal, TypedDict
import uuid
from pprint import pprint

from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver


def pretty_print(obj):
    """Pretty print dictionaries and other objects"""
    pprint(obj, width=100, depth=None)


# Define the shared graph state
class State(TypedDict):
    user_prompt: str
    llm_output: str
    decision: str

In [ ]:
# Generate content using the LLM
def generate_llm_output(state: State) -> State:
    """Generate content using the LLM"""
    try:
        prompt = state.get("user_prompt", "Write a short story about a robot")
        response = llm.invoke(prompt)
        generated_text = response.content if hasattr(response, 'content') else str(response)
        return {"llm_output": generated_text}
    except Exception as e:
        return {"llm_output": f"Error generating content: {str(e)}"}

In [ ]:
# Human approval node
def human_approval(state: State) -> Command[Literal["approved_path", "rejected_path"]]:
    """Present content to human for approval"""
    decision = interrupt({
        "question": "Do you approve the following output?",
        "llm_output": state["llm_output"]
    })

    if decision == "approve":
        return Command(goto="approved_path", update={"decision": "approved"})
    else:
        return Command(goto="rejected_path", update={"decision": "rejected"})

In [ ]:
# Conditional path nodes
def approved_node(state: State) -> State:
    print("Approved path taken.")
    return state

def rejected_node(state: State) -> State:
    print("Rejected path taken.")
    return state

In [ ]:
# Build the graph
builder = StateGraph(State)
builder.add_node("generate_llm_output", generate_llm_output)
builder.add_node("human_approval", human_approval)
builder.add_node("approved_path", approved_node)
builder.add_node("rejected_path", rejected_node)

builder.set_entry_point("generate_llm_output")
builder.add_edge("generate_llm_output", "human_approval")
builder.add_edge("approved_path", END)
builder.add_edge("rejected_path", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from langchain_core.runnables.graph import MermaidDrawMethod
display(
    Image(
        graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
initial_state = {"user_prompt": "Write one line introduction to machine learning for beginners"}

for event in graph.stream(initial_state, config, stream_mode="values"):
    pretty_print(event)

In [ ]:
state = graph.get_state(config)
pretty_print(state)
state.next

In [ ]:
# Simulate a human decision (change to "reject" to test the other branch)
human_decision = "approve"
print(f"Simulating human decision: {human_decision}")

final_result = graph.invoke(Command(resume=human_decision), config=config)
print(f"Final result: {final_result}")

In [ ]:
print(f"Decision made: {final_result.get('decision', 'N/A')}")
print(f"LLM Output: {final_result.get('llm_output', 'N/A')[:100]}...")

## 3. The Edit/Review Pattern

Unlike Approve/Reject (which branches to different nodes), Edit/Review lets a human directly modify the LLM's output before it continues down a single path.

In [ ]:
# Define the graph state for this pattern
class State(TypedDict):
    summary: str


# Generate a summary using the LLM
def generate_summary(state: State) -> State:
    """Generate a summary using the LLM"""
    prompt = "Write a brief summary about a cat sitting on a mat."
    try:
        response = llm.invoke(prompt)
        return {"summary": response.content}
    except Exception as e:
        print(f"Error calling LLM: {e}")
        # Fallback to a hardcoded summary
        return {"summary": "The cat sat on the mat and looked at the stars."}

In [ ]:
# Human editing node
def human_review_edit(state: State) -> State:
    """Allow a human to review and edit the AI-generated summary"""
    result = interrupt({
        "task": "Please review and edit the generated summary if necessary.",
        "generated_summary": state["summary"]
    })
    return {
        "summary": result["edited_summary"]
    }


# Downstream node that consumes the (possibly edited) summary
def downstream_use(state: State) -> State:
    """Process the final summary"""
    print(f"Using edited summary: {state['summary']}")
    return state

In [ ]:
# Build the graph
def create_summary_review_graph():
    """Create and return the configured LangGraph"""
    builder = StateGraph(State)

    builder.add_node("generate_summary", generate_summary)
    builder.add_node("human_review_edit", human_review_edit)
    builder.add_node("downstream_use", downstream_use)

    builder.set_entry_point("generate_summary")
    builder.add_edge("generate_summary", "human_review_edit")
    builder.add_edge("human_review_edit", "downstream_use")
    builder.add_edge("downstream_use", END)

    checkpointer = MemorySaver()
    return builder.compile(checkpointer=checkpointer)


graph = create_summary_review_graph()
display(
    Image(
        graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Step 1: Generating AI summary...")
result = graph.invoke({}, config=config)
print(result)

In [ ]:
interrupt_data = result["__interrupt__"]
interrupt_data

In [ ]:
# Simulate human editing (in a real application, this would be user input)
print("Step 2: Human reviewing and editing...")
edited_summary = "The cat lay on the rug, gazing peacefully at the night sky."

In [ ]:
print("Step 3: Resuming workflow with edited summary...")
final_result = graph.invoke(
    Command(resume={"edited_summary": edited_summary}),
    config=config
)
print(final_result)

## 4. Reviewing Tool Calls Before Execution

A human can review and edit the output from the LLM before proceeding — particularly critical when tool calls requested by the LLM may be sensitive or require oversight.

To add a human approval step to a tool:
1. Use `interrupt()` inside the tool to pause execution.
2. Resume with `Command(resume=...)` to continue based on human input.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langgraph.prebuilt import create_react_agent

# An example of a sensitive tool that requires human review / approval
def book_hotel(hotel_name: str):
    """Book a hotel"""
    response = interrupt(  # (1)
        f"Trying to call `book_hotel` with args {{'hotel_name': {hotel_name}}}. "
        "Please approve or suggest edits."
    )
    if response["type"] == "accept":
        pass
    elif response["type"] == "edit":
        hotel_name = response["args"]["hotel_name"]
    else:
        raise ValueError(f"Unknown response type: {response['type']}")
    return f"Successfully booked a stay at {hotel_name}."

checkpointer = InMemorySaver()  # (2) stores agent state at every step; enables short-term memory + HITL

agent = create_react_agent(
    model=llm,
    tools=[book_hotel],
    checkpointer=checkpointer,  # (3)
)

# (1) interrupt() pauses the agent graph at the node executing the tool. The information
#     inside interrupt() (e.g., tool call args) can be presented to a human, and the graph
#     resumed with the user's input (approval, edit, or feedback).

In [ ]:
config = {
   "configurable": {
      "thread_id": "1"
   }
}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "book a stay at McKittrick hotel"}]},
    config
):
    print(chunk)
    print("\n")

# The agent runs until it reaches interrupt(), then pauses and waits for human input.

In [ ]:
from langgraph.types import Command

# Resume the agent with a Command(resume=...) based on human input
for chunk in agent.stream(
    Command(resume={"type": "accept"}),
    # Command(resume={"type": "edit", "args": {"hotel_name": "McKittrick Hotel"}}),
    config
):
    print(chunk)
    print("\n")

### Generic Wrapper: Add Human Review to Any Tool

Instead of hand-writing `interrupt()` inside every sensitive tool, `add_human_in_the_loop` wraps *any* tool to add a review step around it. The example below is a reference implementation compatible with Agent Inbox UI and Agent Chat UI.

In [ ]:
from typing import Callable
from langchain_core.tools import BaseTool, tool as create_tool
from langchain_core.runnables import RunnableConfig
from langgraph.types import interrupt
from langgraph.prebuilt.interrupt import HumanInterruptConfig, HumanInterrupt

def add_human_in_the_loop(
    tool: Callable | BaseTool,
    *,
    interrupt_config: HumanInterruptConfig = None,
) -> BaseTool:
    """Wrap a tool to support human-in-the-loop review."""
    if not isinstance(tool, BaseTool):
        tool = create_tool(tool)

    if interrupt_config is None:
        interrupt_config = {
            "allow_accept": True,
            "allow_edit": True,
            "allow_respond": True,
        }

    @create_tool(  # (1)
        tool.name,
        description=tool.description,
        args_schema=tool.args_schema
    )
    def call_tool_with_interrupt(config: RunnableConfig, **tool_input):
        request: HumanInterrupt = {
            "action_request": {
                "action": tool.name,
                "args": tool_input
            },
            "config": interrupt_config,
            "description": "Please review the tool call"
        }
        response = interrupt([request])[0]  # (2)
        if response["type"] == "accept":
            tool_response = tool.invoke(tool_input, config)
        elif response["type"] == "edit":
            tool_input = response["args"]["args"]
            tool_response = tool.invoke(tool_input, config)
        elif response["type"] == "response":
            user_feedback = response["args"]
            tool_response = user_feedback
        else:
            raise ValueError(f"Unsupported interrupt response type: {response['type']}")

        return tool_response

    return call_tool_with_interrupt

# (1) This wrapper creates a new tool that calls interrupt() BEFORE executing the wrapped tool.
# (2) interrupt() uses the input/output format expected by Agent Inbox UI: a list of
#     HumanInterrupt objects is sent for rendering, and the resume value comes back as a list
#     (i.e., Command(resume=[...])).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

checkpointer = InMemorySaver()

def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name}."


agent = create_react_agent(
    model=llm,
    tools=[
        add_human_in_the_loop(book_hotel),  # (1) adds interrupt() without modifying book_hotel itself
    ],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "1"}}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "book a stay at McKittrick hotel"}]},
    config
):
    print(chunk)
    print("\n")

In [ ]:
from langgraph.types import Command

for chunk in agent.stream(
    Command(resume=[{"type": "accept"}]),
    # Command(resume=[{"type": "edit", "args": {"args": {"hotel_name": "McKittrick Hotel"}}}]),
    config
):
    print(chunk)
    print("\n")

## Summary

- **`interrupt_before`** on a tool-calling agent lets you inspect or edit conversational state before every LLM turn; a dedicated `human_feedback` node makes the pause point explicit in the graph.
- **Approve/Reject** uses `Command(goto=...)` to branch the graph based on a human decision returned from `interrupt()`.
- **Edit/Review** lets a human directly modify LLM output that flows into a single downstream path.
- **Tool-call review** puts `interrupt()` inside (or around, via `add_human_in_the_loop`) a sensitive tool so a human can accept, edit, or reject the call before it executes.

See **`01_HITL_Mechanics.ipynb`** for the underlying `interrupt()`/`interrupt_before`/`interrupt_after` mechanics these patterns are built from.